# Day 080 — Exercise 4: The ReAct Loop

**What you'll build:** `run_react_agent` — reason, act, observe, repeat.

**Why it matters:** this is the loop that makes ReAct work. It keeps a scratchpad, appends each `Thought/Action/Input` and its `Observation`, and feeds the whole thing back so the model reasons over its own trail — until a `Final Answer` or `max_iterations`.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing a
    runaway loop (a model that never emits a Final Answer).
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── tools reused from Day 79: a safe calculator + a fact-lookup tool ──────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate arithmetic without eval() (see Day 79)."""
    return _eval_node(ast.parse(expression, mode="eval").body)


_FACTS = {
    "speed of light": "299792458 m/s",
    "pi": "3.14159",
    "earth radius": "6371 km",
    "days in a year": "365",
}


def _lookup(args):
    query = str(args.get("query", "")).lower().strip()
    for key, value in _FACTS.items():
        if query and (query in key or key in query):
            return value
    return "No result found for " + repr(args.get("query", ""))


DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "lookup": {
        "description": "Look up a known fact: speed of light, pi, earth radius, "
                       "days in a year.",
        "parameters": {"query": "string - what to look up"},
        "fn": _lookup,
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as prompt text (Day 79)."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None

# ── parsing the ReAct format ──────────────────────────────────────────────────
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive), else ''."""
    for line in text.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):].strip()
    return ""


def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    idx = text.lower().find(marker.lower())
    if idx == -1:
        return None
    return text[idx + len(marker):].strip()


def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.

    Returns either:
      {"type": "action", "thought": str, "tool": str, "input": dict}
      {"type": "final",  "thought": str, "answer": str}
    A reply with no recognisable Action falls back to a final answer holding
    the raw text - so a malformed step still ends the loop cleanly.
    """
    thought = _line_value(text, "Thought:")
    final = _after_marker(text, "Final Answer:")
    if final is not None:
        return {"type": "final", "thought": thought, "answer": final}
    action = _line_value(text, "Action:")
    if action:
        args = safe_parse_json(_line_value(text, "Input:")) or {}
        return {"type": "action", "thought": thought, "tool": action, "input": args}
    return {"type": "final", "thought": thought, "answer": text.strip()}

# ── formatting the trace (the scratchpad) ─────────────────────────────────────
def format_step(step):
    """Render an action step back into ReAct text for the scratchpad."""
    return ("Thought: " + step["thought"] + "\n"
            + "Action: " + step["tool"] + "\n"
            + "Input: " + json.dumps(step["input"]))


def format_observation(result):
    """Render a tool result as an Observation line."""
    return "Observation: " + str(result)


def build_react_prompt(task, tools, scratchpad):
    """Build the [system, user] messages for one ReAct step."""
    system = "\n".join([
        "You are a reasoning agent. Solve the task step by step using the "
        "ReAct format: reason, act, observe, repeat.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "On each turn reply in EXACTLY this format:",
        "Thought: <your reasoning about what to do next>",
        "Action: <one tool name from the list above>",
        'Input: {"<param>": "<value>"}',
        "",
        "You will then receive an Observation with the tool's result.",
        "When you can answer, reply instead with:",
        "Thought: <your final reasoning>",
        "Final Answer: <the answer>",
    ])
    user = "Task: " + str(task)
    if scratchpad:
        user = user + "\n\n" + scratchpad.rstrip()
    user = user + "\n\nThought:"
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]

# ── acting + calling the model ────────────────────────────────────────────────
def execute_action(step, tools):
    """Run the tool named in a ReAct action step. Returns a string, never raises."""
    name = step.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](step.get("input", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Task

`run_react_agent(task, tools=None, llm_fn=None, max_iterations=10) -> dict`

- default `tools` to `DEFAULT_TOOLS`; start `scratchpad = ''`, `trace = []`
- loop `for i in range(max_iterations)`: `build_react_prompt(task, tools, scratchpad)` → `call_llm` → `parse_react_step`
- on a `final` step: append to trace, return `{answer, thought, trace, iterations: i+1, stopped: False}`
- else: `execute_action`, store the result on `step['observation']`, append to trace, and add `format_step(step)` + `format_observation(result)` to the scratchpad
- if the loop ends: return `stopped: True`

## Your Implementation

In [ ]:
def run_react_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the ReAct loop. Returns {answer, thought, trace, iterations, stopped}."""
    raise NotImplementedError


In [ ]:

# ── the ReAct loop ────────────────────────────────────────────────────────────
def run_react_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the ReAct loop until a Final Answer or max_iterations.

    Each turn: build a prompt from the task + scratchpad, ask the model for a
    Thought/Action/Input, run the tool, append the step and its Observation to
    the scratchpad, repeat. Feeding the growing scratchpad back is what lets the
    model reason over its own earlier observations.

    Returns {"answer", "thought", "trace", "iterations", "stopped"}.
    """
    if tools is None:
        tools = DEFAULT_TOOLS
    scratchpad = ""
    trace = []
    for i in range(max_iterations):
        messages = build_react_prompt(task, tools, scratchpad)
        step = parse_react_step(call_llm(messages, llm_fn=llm_fn))
        if step["type"] == "final":
            trace.append(step)
            return {"answer": step["answer"], "thought": step["thought"],
                    "trace": trace, "iterations": i + 1, "stopped": False}
        result = execute_action(step, tools)
        step["observation"] = result
        trace.append(step)
        scratchpad = scratchpad + format_step(step) + "\n"
        scratchpad = scratchpad + format_observation(result) + "\n"
    return {"answer": "Stopped: reached max_iterations without a final answer.",
            "thought": "", "trace": trace,
            "iterations": max_iterations, "stopped": True}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    script = ['Thought: I should add.\nAction: calculator\nInput: {"expression": "2+2"}',
              'Thought: I know now.\nFinal Answer: The answer is 4.']
    out = run_react_agent('what is 2+2', DEFAULT_TOOLS, llm_fn=_make_mock_llm(script))
    assert out['answer'] == 'The answer is 4.' and out['stopped'] is False
    score += 1; print("✅ run_react_agent loops: act, observe, then finish")

    assert len(out['trace']) == 2 and out['trace'][0]['type'] == 'action'
    assert out['trace'][0]['observation'] == '4'
    score += 1; print("✅ trace records each step with its observation")

    # the observation must be fed back into the next prompt
    seen = []
    def _spy(messages):
        seen.append(messages[1]['content'])
        if len(seen) == 1:
            return 'Thought: add.\nAction: calculator\nInput: {"expression": "2+2"}'
        return 'Thought: done.\nFinal Answer: 4'
    run_react_agent('2+2', DEFAULT_TOOLS, llm_fn=_spy)
    assert 'Observation: 4' in seen[1]
    score += 1; print("✅ the scratchpad (with Observation) is fed back")

    assert 'Observation' not in seen[0] and 'Task:' in seen[0]
    score += 1; print("✅ first turn has the task but no observations yet")

    never = _make_mock_llm(['Thought: loop.\nAction: calculator\nInput: {"expression": "1+1"}'])
    loop = run_react_agent('x', DEFAULT_TOOLS, llm_fn=never, max_iterations=3)
    assert loop['stopped'] is True and loop['iterations'] == 3
    score += 1; print("✅ max_iterations stops a runaway loop")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── the ReAct loop ────────────────────────────────────────────────────────────
def run_react_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the ReAct loop until a Final Answer or max_iterations.

    Each turn: build a prompt from the task + scratchpad, ask the model for a
    Thought/Action/Input, run the tool, append the step and its Observation to
    the scratchpad, repeat. Feeding the growing scratchpad back is what lets the
    model reason over its own earlier observations.

    Returns {"answer", "thought", "trace", "iterations", "stopped"}.
    """
    if tools is None:
        tools = DEFAULT_TOOLS
    scratchpad = ""
    trace = []
    for i in range(max_iterations):
        messages = build_react_prompt(task, tools, scratchpad)
        step = parse_react_step(call_llm(messages, llm_fn=llm_fn))
        if step["type"] == "final":
            trace.append(step)
            return {"answer": step["answer"], "thought": step["thought"],
                    "trace": trace, "iterations": i + 1, "stopped": False}
        result = execute_action(step, tools)
        step["observation"] = result
        trace.append(step)
        scratchpad = scratchpad + format_step(step) + "\n"
        scratchpad = scratchpad + format_observation(result) + "\n"
    return {"answer": "Stopped: reached max_iterations without a final answer.",
            "thought": "", "trace": trace,
            "iterations": max_iterations, "stopped": True}
```

**Why keep a text scratchpad and not just the trace list?** The model reads text. The scratchpad is the ReAct transcript the model sees each turn; the `trace` list is the structured version *you* inspect afterwards. Same events, two audiences.

</details>